# ACIT4620 — Notebook 12: Final Locked-Test Evaluation

**Group 1 | HAM10000 | Frozen EfficientNet-B0 + Linear/MLP/First-order Sugeno ANFIS**

## Important research protocol

This notebook intentionally separates **(1) preparation and model-selection freeze** from **(2) opening the locked test partition**. It does not train new models. The frozen-model selection was conducted on the **1,441-image validation set only** (Notebook 09), not on the held-out 1,458-image test set. The package contains the original E2a-50 split manifest in a sealed ZIP; do **not** manually open the ZIP until the team has signed off on the freeze. The app will be built after this research-stage evaluation.

- **Six pre-selected primary models:** CNN; Linear seed 42; MLP seed 42; ANFIS R4 seed 52; ANFIS R8 seed 42; ANFIS R16 seed 52.
- **Secondary, pre-registered sensitivity analysis:** all 16 existing CNN / 3-seed E3 / 3-seed × 3-rule-count E4 checkpoints, evaluated in the *same locked pass*.
- **Primary metric:** macro-F1. Also balanced accuracy, melanoma recall, per-class values, probabilistic scores, lesion-cluster paired 95% bootstrap intervals.
- **Do not report human-verified acquisition-artifact results yet:** the 50-image annotation stage was intentionally deferred.

Read `MODEL_SELECTION_AND_EVALUATION_PROTOCOL.md` and `TEAM_SIGNOFF_BEFORE_TEST.md` before unsealing.

## 1. Project paths and Python environment

Edit paths only to point to your **existing original** repository and Notebook11 model-asset folders. No weights are bundled with Notebook12 and no retraining is allowed.

In [ ]:
from pathlib import Path
import subprocess, sys, json, os

# Find the repository root from the notebook working directory.
repo_override = os.environ.get("ACIT4620_REPO_ROOT")

if repo_override:
    REPO_ROOT = Path(repo_override).expanduser().resolve()
else:
    cwd = Path.cwd().resolve()
    REPO_ROOT = next(
        (
            p for p in (cwd, *cwd.parents)
            if (
                p / "evaluation"
                / "notebook12_locked_test"
                / "locked_test.py"
            ).is_file()
        ),
        None
    )

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, "
        "or configure ACIT4620_REPO_ROOT."
    )

NB12_ROOT = (
    REPO_ROOT / "evaluation" / "notebook12_locked_test"
)

TEAM_REPO = REPO_ROOT

# External original experiment assets must be configured locally.
CODE_ROOT = Path(
    os.environ.get(
        "ACIT4620_CODE_ROOT",
        "CHANGE_ME_TO_YOUR_CODE_FOLDER"
    )
).expanduser()

BASE = CODE_ROOT.parent

NB11_ROOT = (
    CODE_ROOT
    / "Notebook_11_Computational_Efficiency_VSCode_Complete"
    / "ACIT4620_11_Efficiency_VSCode"
)

RAW_IMAGES_ROOT = (
    CODE_ROOT
    / "Notebook_10_CNN_ANFIS_XAI_VSCode_Complete"
    / "ACIT4620_10_CNN_ANFIS_XAI_VSCode"
    / "data"
    / "raw"
)

PYTHON = Path(sys.executable)

# Use the ORIGINAL pre-test freeze when evaluating.
# It is intentionally not committed to GitHub.
FREEZE_JSON = Path(
    os.environ.get(
        "ACIT4620_FREEZE_JSON",
        str(NB12_ROOT / "outputs" / "pretest_freeze.json")
    )
).expanduser()

OUTPUTS = (
    NB12_ROOT / "outputs" / "ACIT4620_12_Locked_Test_Results"
)

checks = {
    "Notebook12 script": NB12_ROOT / "locked_test.py",
    "Original E3 linear checkpoint": TEAM_REPO / "experiments" / "e3_linear_mlp" / "outputs" / "ACIT4620_E3_Linear_MLP" / "linear_seed42" / "best_head.pth",
    "Original Notebook11 ANFIS checkpoint": NB11_ROOT / "data" / "model_assets" / "anfis_r16_seed52.pth",
    "Original raw HAM10000 image root": RAW_IMAGES_ROOT,
    "Working Python virtual environment": PYTHON,
}
for label,path in checks.items():
    print(f"{'PASS' if path.exists() else 'MISSING'}: {label}\n  {path}")
assert all(p.exists() for p in checks.values()), "Fix the MISSING original paths. Do NOT make replacement model weights."
print("PASS: local original resource paths are resolved. No test data opened.")

## 2. Unit tests — safe to run before freeze

These tests use **synthetic labels only** and verify the exact Notebook09 selection anchor, seven-class metrics, paired lesion resampling and tamper detection. They do not open held-out images.

In [ ]:
res = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", str(NB12_ROOT / "tests"), "-v"], check=True)
print("PASS: pure-synthetic unit tests.")

## 3. Freeze exact model-selection and scoring protocol (NO TEST ACCESS)

Execute this cell **before** the final test run. It validates all original 16 head/CNN checkpoint files and both train-only scalers, and freezes their SHA-256 fingerprints. It **does not read the sealed test manifest or decode images**. It intentionally refuses to overwrite an existing freeze. If the source assets are missing, restore your exact training outputs from their original location.

In [ ]:
assert not FREEZE_JSON.exists(), (
    f"A freeze already exists: {FREEZE_JSON}. Do not overwrite a pre-test freeze."
)
subprocess.run([
    str(PYTHON), str(NB12_ROOT / "locked_test.py"), "freeze",
    "--repo", str(TEAM_REPO), "--nb11", str(NB11_ROOT),
    "--freeze", str(FREEZE_JSON)
], check=True)
print("PASS: freeze generated; TEST LABELS AND IMAGES WERE NOT ACCESSED.")

## 4. Inspect the frozen record and obtain real team sign-off

Review the predefined six models, expected dataset size, hashes, preprocessing and statistical plan. Do not interpret or access test results before the entire team has agreed to the fixed protocol. Complete `TEAM_SIGNOFF_BEFORE_TEST.md` **manually**, not through auto-filled names.

In [ ]:
assert FREEZE_JSON.exists(), "Freeze not yet generated."
freeze = json.loads(FREEZE_JSON.read_text(encoding="utf-8"))
payload = freeze["payload"]
print("Freeze created:", payload["created_utc"])
print("Primary selected models:")
for model in payload["primary_models"]:
    print(f"  {model:<21} validation macro-F1={payload['validation_macro_f1'][model]:.4f}")
print("Additional fixed checkpoints evaluated in same pass:", len(payload["secondary_all_seed_models"]))
print("Expected untouched test images:", payload["expected_test_images"])
print("Test images/labels touched during freeze?", payload["test_images_or_labels_read_during_freeze"])
print("Original checkpoint count:",len(payload["checkpoint_and_scaler_sha256"]))
print("Freeze digest:",freeze["payload_sha256"])
print("NEXT: sign TEAM_SIGNOFF_BEFORE_TEST.md with team members.")

## 5. FINAL LOCKED TEST — ONLY AFTER SIGN-OFF

**Deliberately disabled by default.** Change `RUN_FINAL_LOCKED_TEST` to `True` *only once*, after all team members review and sign off on the freeze. This operation opens the original 1,458-image held-out test partition and runs all 16 fixed models in one pass. It refuses to overwrite non-empty results. **Do not tune or choose models using the test scores.**

For CPU inference keep `DEVICE="cpu"`. Set `"cuda"` only when your selected Python environment actually supports CUDA. Batch size changes affect throughput, not model selection.


In [ ]:
RUN_FINAL_LOCKED_TEST = False  # EXPLICITLY change to True only after real team sign-off
DEVICE = "cpu"
BATCH_SIZE = 16

if RUN_FINAL_LOCKED_TEST:
    assert FREEZE_JSON.exists(), "Run freeze and obtain sign-off first"
    assert not (OUTPUTS.exists() and any(OUTPUTS.iterdir())), "Do not overwrite existing locked-test outputs"
    subprocess.run([
        str(PYTHON), str(NB12_ROOT / "locked_test.py"), "evaluate",
        "--repo", str(TEAM_REPO), "--nb11", str(NB11_ROOT),
        "--raw", str(RAW_IMAGES_ROOT), "--freeze", str(FREEZE_JSON),
        "--out", str(OUTPUTS), "--device", DEVICE,
        "--batch-size", str(BATCH_SIZE), "--bootstrap-repeats", "2000",
        "--confirm-locked-test"
    ], check=True)
else:
    print("SAFE: locked test was NOT opened. Review freeze JSON and obtain team sign-off first.")

## 6. Read the completed results (AFTER the locked evaluation only)

This cell only displays an already saved result. It does **not** retrain or rerun the locked test.

In [ ]:
if (OUTPUTS / "six_prespecified_models_test_metrics.csv").exists():
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(OUTPUTS / "six_prespecified_models_test_metrics.csv")[[
        "model", "macro_f1", "balanced_accuracy", "accuracy", "melanoma_recall", "melanoma_fn"
    ]])
    display(pd.read_csv(OUTPUTS / "paired_lesion_bootstrap_intervals.csv"))
    print("Results folder:", OUTPUTS)
else:
    print("No locked-test output exists yet. No test results available. This is expected before sign-off.")

## 7. Report integration and the next implementation phase

After a **single successful** locked-test run, use the saved outputs for the final IEEE report: source/split provenance, pre-registered checkpoint-selection method, six-model test comparison, seed sensitivity, per-class melanoma outcomes, lesion-cluster confidence intervals and limitations. Reference Notebook10/11 results separately, without claiming that the deferred acquisition-artifact annotations were completed.

**Afterward:** package the selected CNN+ANFIS source checkpoints into a reusable inference module and build the image-upload demonstration app with proper research-only/medical disclaimers. No app components or unseen test results are fabricated here.
